# 11.01 · Harness thinking: build a mini-harness, then `create_deep_agent`

**Module 11 · Deep Agents** · Part 2 (Pro)

| | |
|---|---|
| ⏱️ Estimated time | 2–3 h |
| 📋 Prerequisites | Module 09 (subagents, `task_tool`), Lesson 04.03 (custom middleware), Lesson 08.01 (checkpointers). |
| 🔑 Keys | none for 🧪 cells · `GOOGLE_API_KEY` (or the key of your `MASTERY_DEEP_MODEL`) for the 💸 cell, ≈ 8–15K tokens |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-16 |

> **How to use this notebook:** read each Markdown cell, then run the code cell below it with **Shift + Enter**, top to bottom.
> Cell markers: **🧪** offline (scripted model, no network, free) · **🌐** network, no tokens · **💸** calls a model (uses your free-tier quota).
> Without a key, 💸 cells print `⏭️ Skipped` and everything else still runs, so **Kernel → Restart Kernel and Run All Cells…** always completes.

## Learning objectives

1. **Explain** "agent = model + harness" and name the four harness areas the docs use: execution environment, context management, delegation, steering.
2. **Build** a mini-harness from `create_agent` + middleware: file tools over a `files` dict in state, a todo tool whose plan is injected into every model call, and a `task` tool with an isolated subagent.
3. **Build** the same agent with `create_deep_agent`, always passing the model explicitly, and map each built-in piece to the one you wrote.
4. **Measure** what each harness sends per call (system prompt, tool schemas, messages) and **decide** when a deep agent is the wrong tool.

## 📚 Official reading (verified 2026-10-16, `deepagents` 0.7.26)

| Page | Read it for |
|---|---|
| [Deep Agents overview](https://docs.langchain.com/oss/python/deepagents/overview) | What the harness adds: planning, a virtual filesystem, subagents, context management |
| [Quickstart](https://docs.langchain.com/oss/python/deepagents/quickstart) | The smallest `create_deep_agent` program |
| [Customize Deep Agents](https://docs.langchain.com/oss/python/deepagents/customization) | `system_prompt`, `tools`, `middleware`, `subagents` |
| [`create_deep_agent` reference](https://reference.langchain.com/python/deepagents/graph/create_deep_agent) | Every parameter, and the deprecation of `model=None` |
| [Custom middleware](https://docs.langchain.com/oss/python/langchain/middleware/custom) · [Tools: state updates with `Command`](https://docs.langchain.com/oss/python/langchain/tools) | The two building blocks of your mini-harness |
| [Runtimes, frameworks, and harnesses](https://docs.langchain.com/oss/python/concepts/products) · [Deep Agents vs Claude Agent SDK](https://docs.langchain.com/oss/python/deepagents/comparison) | Where a harness sits, and the alternatives |
| [Changelog: `deepagents` v0.7.0](https://docs.langchain.com/oss/python/deepagents/changelog-py) | The leaner default harness (todos opt-in, ≈65% fewer input tokens per default turn) |

---
## Step 0 · Install and set up

In [ ]:
%pip install --quiet "langchain>=1.4.4,<2" "langchain-core>=1.6.9,<2" "langgraph>=1.2.14,<2" "deepagents>=0.7.23,<0.8" "langchain-google-genai>=4.4.1,<5" "langchain-groq>=1.1.3,<2"

**You should now see** *"Note: you may need to restart the kernel to use updated packages."*, possibly below a few lines from pip. IPython prints that note after **every** `%pip`; restart only if a package was **upgraded** after you had already imported it (Lesson 00.05). `%pip` always installs into **this kernel's** environment; `!pip` might not (Lesson 00.02).

In [ ]:
from importlib import metadata

for package in ("deepagents", "langchain", "langgraph", "langchain-google-genai",):
    print(f"{package:<22} {metadata.version(package)}")

In [ ]:
# Setup 1: Find the repository root (Jupyter starts in modules/11-deep-agents/).
import importlib.util
import os
import sys
from pathlib import Path

# Optional: set LC_MASTERY_ROOT to your clone if Jupyter starts outside it (no paths are hardcoded).
_override = os.environ.get("LC_MASTERY_ROOT", "").strip()
REPO_ROOT = next(
    (p for p in ([Path(_override).expanduser()] if _override else [Path.cwd(), *Path.cwd().parents])
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the course repo. Start Jupyter inside your clone (or set LC_MASTERY_ROOT to it).")

# Setup 2: Import `mastery` from THIS clone's src/, like pytest does (pythonpath = ["src"]), and check
# that the env's own install (00.02 guide, Step 5) points here too: terminal commands use that one.
SRC_DIR = (REPO_ROOT / "src").resolve()
installed_spec = importlib.util.find_spec("mastery")
if sys.path[0] != str(SRC_DIR):
    sys.path.insert(0, str(SRC_DIR))
import mastery

print("Repo root:", REPO_ROOT)
print("mastery  :", mastery.__version__, "from", mastery.__file__)
installed_at = installed_spec.origin if installed_spec else None
if not Path(mastery.__file__).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This kernel had already imported mastery from another folder: restart the kernel, run from the top.")
elif installed_at is None or not Path(installed_at).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This env's own mastery install does not use this clone's src/:", installed_at or "(not installed)")
    print("   This notebook uses this clone's src/ anyway, but terminal commands (python -m mastery...) do not.")
    print('   Fix once (Git Bash, course env active, root of THIS clone): python -m pip install -e ".[dev]"')

In [ ]:
KEYS_TO_CHECK = ('GOOGLE_API_KEY', 'GROQ_API_KEY', 'LANGSMITH_API_KEY')
LANGSMITH_PROJECT = 'm11-harness'
# Setup 3: Load .env, check keys WITHOUT printing them, read model IDs from config.
import os

from mastery.config import env_status, is_set, load_env, load_settings
from mastery.deep import deep_key_ready, deep_model_spec, fallback_specs

dotenv_path = load_env()
settings = load_settings(load_dotenv_file=False)   # .env was just loaded.
HAVE_GROQ_KEY = is_set("GROQ_API_KEY")
HAVE_LANGSMITH_KEY = is_set("LANGSMITH_API_KEY")

DEEP_MODEL = deep_model_spec("main")              # MASTERY_DEEP_MODEL (course default: Gemini free tier)
SUBAGENT_MODEL = deep_model_spec("subagent")      # MASTERY_DEEP_SUBAGENT_MODEL
HAVE_DEEP_KEY = deep_key_ready(DEEP_MODEL)        # the key for THAT provider (GOOGLE_API_KEY for google_genai:)
os.environ.setdefault("LANGSMITH_PROJECT", LANGSMITH_PROJECT)   # m11-<lesson>: one tracing project per lesson

print(".env      :", dotenv_path or "not found (fine if keys are set in your shell)")
for name, status in env_status(*KEYS_TO_CHECK).items():
    print(f"{name:<18}: {status}")
print("project   :", os.environ["LANGSMITH_PROJECT"])
print("deep model:", DEEP_MODEL, "| subagents:", SUBAGENT_MODEL)
print("fallbacks :", fallback_specs(DEEP_MODEL) or "none (optional: MASTERY_FALLBACK_MODELS in .env)")
if not HAVE_DEEP_KEY:
    print(f"\n⚠️ No key for {DEEP_MODEL.split(':')[0]}: 💸 cells will print ⏭️ Skipped; 🧪 cells still teach the concept.")
if not HAVE_LANGSMITH_KEY:
    print("⚠️ No LANGSMITH_API_KEY: 🌐 LangSmith cells will print ⏭️ Skipped; offline twins run instead.")

In [ ]:
# Setup 4: Silence known, harmless noise so real problems stand out.
import os
import warnings
from importlib.metadata import version

# langsmith warns once when a Client is created without a key; offline cells never send anything.
warnings.filterwarnings("ignore", message="API key must be provided when using hosted LangSmith API")
# Same as Lesson 02.04: older langchain-core 1.6.x Pydantic serializer warning about the injected runtime.
# Only older releases raise it; on current ones this filter is skipped.
if tuple(int("".join(c for c in x if c.isdigit()) or 0) for x in version("langchain-core").split(".")[:3]) < (1, 6, 11):
    warnings.filterwarnings("ignore", message="Pydantic serializer warnings", category=UserWarning)

In [ ]:
def table(rows, columns):
    """Print dict rows as an aligned text table."""
    widths = {c: max(len(c), *(len(str(r.get(c, ""))) for r in rows)) for c in columns}
    print("  ".join(c.ljust(widths[c]) for c in columns))
    for r in rows:
        print("  ".join(str(r.get(c, "")).ljust(widths[c]) for c in columns))

**You should now see** your deep model (default `google_genai:gemini-3.6-flash`), the subagent model, and `fallbacks: none` unless you set `MASTERY_FALLBACK_MODELS`. No key is fine: every 🧪 cell runs offline.

> ⚠️ **Production note: Deep Agents is pre-1.0.** The course allows `deepagents>=0.7.23,<0.8` so lessons keep working while you learn. In production: pin the exact version in a lock file, read the changelog on every bump, and run your evals (Lesson 11.11) as the upgrade gate. All course code goes through `mastery.deep`, so a breaking change touches one file.

---
## Part 1 · Agent = model + harness

A model only turns messages into a message. Everything else that makes an agent useful over a long task is the **harness**: prompts, tools, middleware and state. The Deep Agents docs group what a harness must provide into four areas:

| Area | The question it answers | In this lesson |
|---|---|---|
| Execution environment | Where does the agent keep and change things? | a `files` dict in state, file tools |
| Context management | How does the model see what matters, without everything? | the plan injected each call; files instead of chat |
| Delegation | How is a sub-task done in a clean context? | a `task` tool with an isolated subagent |
| Steering | How is the agent kept on track? | a plan tool + a middleware that shows the plan |

Nothing here is magic: you will build each piece in ~100 lines, then see the library's version.

## Part 2 · Build a mini-harness

In [ ]:
# 🧪 The mini-harness, part 1: state + file tools that update it by returning Command.
from typing import Annotated, Any

from langchain.agents import AgentState
from langchain.tools import ToolRuntime, tool
from langchain_core.messages import ToolMessage
from langgraph.types import Command
from typing_extensions import NotRequired


def merge_files(old: dict | None, new: dict | None) -> dict:
    """Reducer: a tool returns only the files it changed; the state keeps the rest."""
    return {**(old or {}), **(new or {})}


class MiniState(AgentState):
    files: Annotated[NotRequired[dict[str, str]], merge_files]   # path -> text: the agent's "disk"
    todos: NotRequired[list[dict]]                                 # the plan, written by write_todos


def _reply(runtime: ToolRuntime, text: str, **update: Any) -> Command:
    """Update state AND answer the tool call (every tool call needs its ToolMessage)."""
    return Command(update={**update, "messages": [ToolMessage(text, tool_call_id=runtime.tool_call_id)]})


@tool
def ls(runtime: ToolRuntime) -> str:
    """List the files in the agent's workspace."""
    return "\n".join(sorted(runtime.state.get("files", {}))) or "No files found"


@tool
def read_file(path: str, runtime: ToolRuntime) -> str:
    """Read one file from the workspace."""
    return runtime.state.get("files", {}).get(path, f"Error: {path} not found")


@tool
def write_file(path: str, content: str, runtime: ToolRuntime) -> Command:
    """Create or overwrite a file in the workspace."""
    return _reply(runtime, f"Wrote {path} ({len(content)} chars)", files={path: content})


@tool
def edit_file(path: str, old: str, new: str, runtime: ToolRuntime) -> Command | str:
    """Replace the exact text `old` with `new` in one file (`old` must appear exactly once)."""
    text = runtime.state.get("files", {}).get(path)
    if text is None or text.count(old) != 1:
        return f"Error: `old` must appear exactly once in {path}"
    return _reply(runtime, f"Edited {path}", files={path: text.replace(old, new)})

**What just happened?** The tools change **state**, not just the chat: `write_file` returns a `Command` with `files={path: content}`, and the `merge_files` reducer merges it into the dict (a tool reports only what it changed). The `ToolMessage` in the same `Command` answers the tool call, so the model sees `Wrote /notes.md`.

In [ ]:
# 🧪 The mini-harness, part 2: a plan tool, a middleware that shows the plan on EVERY call, a task tool.
from langchain.agents import create_agent
from langchain.agents.middleware import AgentMiddleware
from langchain_core.messages import SystemMessage

from mastery.agents.specialists import task_tool   # Module 09: one dispatch tool, isolated subagent context


@tool
def write_todos(todos: list[str], runtime: ToolRuntime) -> Command:
    """Replace the plan: a short list of steps. Mark finished steps with a leading [x]."""
    items = [{"step": t.removeprefix("[x]").strip(), "done": t.startswith("[x]")} for t in todos]
    return _reply(runtime, f"Plan has {len(items)} steps", todos=items)


class TodoReminder(AgentMiddleware):
    """Steering: the plan lives in state, not in the chat, so we add it to the system prompt each call."""

    state_schema = MiniState

    def wrap_model_call(self, request, handler):
        todos = request.state.get("todos") or []
        if todos:
            plan = "\n".join(f"- [{'x' if t['done'] else ' '}] {t['step']}" for t in todos)
            base = request.system_message.text if request.system_message else ""
            request = request.override(system_message=SystemMessage(f"{base}\n\nCurrent plan:\n{plan}".strip()))
        return handler(request)


def build_mini_harness(model, sub_model, system_prompt: str, middleware=()):
    researcher = create_agent(sub_model, tools=[], system_prompt="Answer the task in one or two sentences.")
    task = task_tool({"researcher": (researcher, "Looks up one fact and answers in one or two sentences.")})
    return create_agent(model, tools=[ls, read_file, write_file, edit_file, write_todos, task],
                        middleware=[TodoReminder(), *middleware], system_prompt=system_prompt)


print("mini-harness tools:", [t.name for t in (ls, read_file, write_file, edit_file, write_todos)], "+ task")

**What just happened?** Steering and delegation in two small pieces. `TodoReminder` reads `todos` from state on **every** model call and appends them to the system prompt: the plan stays in view even after 30 turns, without repeating it in the chat. `task_tool` (Module 09) gives the subagent a **fresh** conversation containing only the task, so its tool noise never reaches the main agent's context.

In [ ]:
# 🧪 A tiny meter: what does each model call SEND? (system prompt + tool schemas + messages, ≈ chars / 4)
import json

from langchain_core.utils.function_calling import convert_to_openai_tool


class PromptMeter(AgentMiddleware):
    """Records an estimate of the input size of every model call (put it LAST: it sees the final request)."""

    def __init__(self):
        super().__init__()
        self.calls = []

    def wrap_model_call(self, request, handler):
        system = request.system_message.text if request.system_message else ""
        tools = sum(len(json.dumps(convert_to_openai_tool(t))) for t in request.tools if not isinstance(t, dict))
        chat = sum(len(str(m.content)) for m in request.messages)
        self.calls.append({"system": len(system) // 4, "tools": tools // 4, "messages": chat // 4})
        return handler(request)

    def summary(self, name):
        first = self.calls[0] if self.calls else {}
        return {"agent": name, "calls": len(self.calls), "system≈tok": first.get("system", 0),
                "tool schemas≈tok": first.get("tools", 0),
                "total input≈tok": sum(sum(c.values()) for c in self.calls)}

In [ ]:
TASK = "Write /notes.md: a 2-line note on what a LangGraph checkpointer does. Plan first, delegate the research."
SYSTEM = "You are a careful assistant. Keep files short."

In [ ]:
# 🧪 Offline: run the mini-harness on a scripted model (no network, free).
from langchain_core.messages import AIMessage

from mastery.testing import ScriptedChatModel

main_script = ScriptedChatModel(responses=[
    AIMessage("", tool_calls=[{"name": "write_todos", "args": {"todos": ["Research checkpointers", "Write the note"]}, "id": "t1"}]),
    AIMessage("", tool_calls=[{"name": "task", "args": {"agent_name": "researcher",
                                                     "description": "What does a LangGraph checkpointer do?"}, "id": "t2"}]),
    AIMessage("", tool_calls=[{"name": "write_file", "args": {"path": "/notes.md",
                                                           "content": "# Checkpointers\nThey save graph state after every step."}, "id": "t3"}]),
    AIMessage("", tool_calls=[{"name": "write_todos", "args": {"todos": ["[x] Research checkpointers", "[x] Write the note"]}, "id": "t4"}]),
    AIMessage("Done: see /notes.md")])
sub_script = ScriptedChatModel(responses=[AIMessage("A checkpointer saves graph state after every step.")])

mini_meter = PromptMeter()
mini = build_mini_harness(main_script, sub_script, SYSTEM, middleware=[mini_meter])
mini_result = mini.invoke({"messages": [{"role": "user", "content": TASK}]})
print("files:", mini_result["files"])
print("todos:", mini_result["todos"])
print("\nWhat the model saw on call 3 (the plan was injected):\n" + main_script.calls[2][0].content)

**You should now see** `files: {'/notes.md': ...}`, both todos with `done: True`, and the system prompt of call 3 ending with `Current plan:` and two unchecked steps. That is the plan injected by your middleware, after `write_todos` (call 1) and the delegation (call 2).

## Part 3 · The same agent with `create_deep_agent`

`build_deep_agent()` from `mastery.deep` calls `create_deep_agent(model=..., tools=..., system_prompt=..., middleware=...)` with the course defaults: an **explicit model** (`model=None` is deprecated since 0.5.3), a rate limiter, per-run call budgets, and optional fallbacks.

In [ ]:
# 🧪 Offline: the same task on create_deep_agent (through mastery.deep.build_deep_agent).
from mastery.deep import build_deep_agent, run_stats

deep_script = ScriptedChatModel(responses=[   # ONE model: the general-purpose subagent reuses the main model
    AIMessage("", tool_calls=[{"name": "task", "args": {"subagent_type": "general-purpose",
                                                     "description": "What does a LangGraph checkpointer do? Two sentences."}, "id": "d1"}]),
    AIMessage("A checkpointer saves graph state after every step."),        # <- the subagent's answer
    AIMessage("", tool_calls=[{"name": "write_file", "args": {"file_path": "/notes.md",
                                                           "content": "# Checkpointers\nThey save graph state after every step."}, "id": "d2"}]),
    AIMessage("Done: see /notes.md")])

deep_meter = PromptMeter()
deep = build_deep_agent(deep_script, system_prompt=SYSTEM, middleware=[deep_meter], fallbacks=False)
deep_result = deep.invoke({"messages": [{"role": "user", "content": TASK}]})
print("type :", type(deep).__name__, "(a compiled LangGraph graph: invoke, stream, checkpointers as in Module 08)")
print("tools:", deep_script.bound_tools)
print("files:", {path: data["content"] for path, data in deep_result["files"].items()})
print("stats:", run_stats(deep_result))

**You should now see** `CompiledStateGraph`, the built-in tools `['ls', 'read_file', 'write_file', 'edit_file', 'delete', 'glob', 'grep', 'task']` (no `write_todos`: planning is opt-in since v0.7), the note in `files` (each file also has `encoding`, `created_at`, `modified_at`), and `model_calls: 3` for the main agent (the subagent's call is inside the `task` tool).

| You wrote | Deep Agents built-in | What the built-in adds |
|---|---|---|
| `ls`, `read_file`, `write_file`, `edit_file` + `files` reducer | `FilesystemMiddleware` | `delete`, `glob`, `grep`, paginated reads, large tool results offloaded to files, pluggable **backends** (Lesson 11.04), permissions |
| `write_todos` + `TodoReminder` | `TodoListMiddleware` (opt-in) | `pending` / `in_progress` / `completed` states, a tuned prompt (Lesson 11.03) |
| `task_tool` | `SubAgentMiddleware` (`task`) | a general-purpose subagent by default, custom subagents, shared files (Lesson 11.05) |
| nothing | `SummarizationMiddleware` | compacts old history when the context fills up (Lesson 11.06) |
| nothing | `PatchToolCallsMiddleware` | repairs dangling tool calls (e.g. after an interrupt) so the history stays valid |
| nothing | `execute` | shell commands, only with a sandbox backend (Lesson 11.09) |

The prompt is assembled in this order: **your `system_prompt`** → the model's **harness-profile base prompt** → the profile suffix (Lesson 11.02 shows profiles). Since v0.7 the authored base prompt is empty, so for this scripted model the system prompt the model receives is exactly `SYSTEM`: the harness speaks through its **tool schemas**.

## Part 4 · What does each harness send per call?

In [ ]:
# 🧪 Compare what each harness sends: plain create_agent (no tools), the mini-harness, the deep agent.
plain_meter = PromptMeter()
plain = create_agent(ScriptedChatModel(responses=[AIMessage("A checkpointer saves state.")]), tools=[],
                     system_prompt=SYSTEM, middleware=[plain_meter])
plain.invoke({"messages": [{"role": "user", "content": TASK}]})

table([plain_meter.summary("create_agent (no tools)"), mini_meter.summary("mini-harness"),
       deep_meter.summary("create_deep_agent")],
      ["agent", "calls", "system≈tok", "tool schemas≈tok", "total input≈tok"])

**What just happened?** Measured with the same `PromptMeter`, last in each stack, so it sees the final request. The deep agent's first call is bigger than plain `create_agent`'s because of its **tool schemas** (eight tools); v0.7 trimmed them to ≈1,900 input tokens for a default turn, from ≈5,400 in v0.6 (changelog). The scripted numbers are estimates (characters / 4); the 💸 cell below shows real token counts. Per-call overhead is multiplied by every step and every subagent: on a free tier, **steps are the budget**.

> ⚠️ **Production note: estimate before you run.** Characters / 4 is a rough rule. In production: read real `usage_metadata` (or LangSmith cost tracking) per run, set `ModelCallLimitMiddleware` / `ToolCallLimitMiddleware`, and alert on cost per successful task (Lesson 11.11).

## Part 5 · 💸 Run it on a real model

> 💸 Calls your deep model (`MASTERY_DEEP_MODEL`, default Gemini free tier). Rate-limited to one call every 4 s and capped per run. Without the key it prints `⏭️ Skipped`.

In [ ]:
# 💸 LIVE: the same task on your deep model, mini-harness then deep agent. ≈ 8–15K tokens in total.
if HAVE_DEEP_KEY:
    from mastery.deep import budget_middleware, deep_model, fallback_middleware

    backup = fallback_middleware(DEEP_MODEL)          # MASTERY_FALLBACK_MODELS, if you set it
    extra = [backup] if backup else []
    live_rows = []
    for name, make in [
        ("mini-harness", lambda m: build_mini_harness(deep_model("main"), deep_model("subagent"), SYSTEM,
                                                      middleware=[*budget_middleware(12, 20), *extra, m])),
        ("create_deep_agent", lambda m: build_deep_agent(system_prompt=SYSTEM, model_calls=12, tool_calls=20,
                                                         middleware=[m])),
    ]:
        meter = PromptMeter()
        result = make(meter).invoke({"messages": [{"role": "user", "content": TASK}]})
        files = result.get("files", {})
        note = files.get("/notes.md")
        note = note["content"] if isinstance(note, dict) else note
        live_rows.append({**meter.summary(name), **{k: v for k, v in run_stats(result).items() if k.endswith("tokens")}})
        print(f"--- {name}: /notes.md ---\n{note or '(not written: read the last message below)'}")
        print("last message:", str(result["messages"][-1].content)[:200], "\n")
    table(live_rows, ["agent", "calls", "total input≈tok", "input_tokens", "output_tokens"])
else:
    print(f"⏭️ Skipped: no key for {DEEP_MODEL} (see the setup cell).")

**You should now see** two notes and a table with real `input_tokens` / `output_tokens`. Your numbers vary run to run: a real model may skip the plan, delegate twice, or answer from memory. Read the trace in LangSmith (project `m11-harness`) if `LANGSMITH_API_KEY` is set: each `task` call is a nested run.

## Part 6 · When a deep agent is the wrong tool

| Situation | Better choice | Why |
|---|---|---|
| One question, one or two tool calls | `create_agent` (Module 02) | The harness overhead is paid on every call for features you don't use |
| A fixed, known sequence of steps | a LangGraph workflow (Module 07) | Deterministic, cheaper, easier to test |
| Strict latency budget (sub-second) | a single model call or a small agent | Planning and delegation add round trips |
| Long, open-ended, multi-step work with intermediate artifacts | **a deep agent** | Files, planning, delegation and context management pay off |

## ⚠️ Pitfalls

| Symptom | Cause | Fix |
|---|---|---|
| `DeprecationWarning` about the default model | `create_deep_agent()` called without `model=` | Always pass the model (`build_deep_agent` does) |
| A custom tool's state change is lost | The tool returned a string, not a `Command(update=...)` | Return `Command` with the state key **and** a `ToolMessage` |
| `ValueError` about a missing `ToolMessage` | A `Command` without a `ToolMessage` for the tool call | Add `ToolMessage(..., tool_call_id=runtime.tool_call_id)` |
| No `write_todos` tool | Planning is opt-in since v0.7 | `middleware=[TodoListMiddleware()]` (Lesson 11.03) |
| `429` / `RESOURCE_EXHAUSTED` | Free-tier per-minute or per-day limit | Wait; keep the rate limiter; set `MASTERY_FALLBACK_MODELS` (see `.env.example`) |
| A `.with_fallbacks()` model is rejected | `create_deep_agent` needs a real chat model | Use `ModelFallbackMiddleware` (what `mastery.deep` does) |

## 🏋️ Exercises

1. **(Easy)** Run the mini-harness offline with an `ls` call scripted before `write_file`. What does `ls` return on an empty workspace?
2. **(Medium)** Seed a file: invoke the deep agent with `{"messages": [...], "files": {"/brief.md": create_file_data("Audience: new hires")}}` (`from deepagents.backends.utils import create_file_data`) and script a `read_file` call. Does the subagent see the file too?
3. **(Medium)** Add a `delete_file` tool to the mini-harness. Why can't the `merge_files` reducer delete a key, and how would you change it?
4. **(Stretch)** Opt in to planning: build the deep agent with `middleware=[TodoListMiddleware(), deep_meter2]` (`from langchain.agents.middleware import TodoListMiddleware`) and compare its first call with `deep_meter`'s. What did planning add?

<details>
<summary><b>Show solutions</b></summary>

**1.** `No files found` (the same string the built-in `ls` returns since v0.7).

**2.** Yes for the general-purpose subagent: with the default `StateBackend`, the main agent and its subagents share the thread's files (Lesson 11.03).
```python
from deepagents.backends.utils import create_file_data
seeded = deep.invoke({"messages": [{"role": "user", "content": TASK}], "files": {"/brief.md": create_file_data("Audience: new hires")}})
```

**3.** The reducer only adds or replaces keys. Use a sentinel, e.g. return `files={path: None}` and make the reducer drop keys whose value is `None`:
```python
def merge_files(old, new):
    merged = {**(old or {}), **(new or {})}
    return {k: v for k, v in merged.items() if v is not None}
```

**4.** `write_todos` joins the tools, and `TodoListMiddleware` adds its planning instructions to the system prompt: both `system≈tok` and `tool schemas≈tok` grow on **every** call. That per-call cost is why v0.7 made planning opt-in; Lesson 11.03 tests whether it pays off.
```python
from langchain.agents.middleware import TodoListMiddleware
deep_meter2 = PromptMeter()
deep_script.reset()
build_deep_agent(deep_script, system_prompt=SYSTEM, middleware=[TodoListMiddleware(), deep_meter2],
                 fallbacks=False).invoke({"messages": [{"role": "user", "content": TASK}]})
table([deep_meter.summary("default"), deep_meter2.summary("+ TodoListMiddleware")],
      ["agent", "system≈tok", "tool schemas≈tok"])
```
</details>

## 🔎 Docs drill (5 min)

In `deepagents` 0.7.x, which tools does a default `create_deep_agent` give the model, and which one appears only with a sandbox backend?

Try it before you open the answer: use the official docs, the API reference, or the installed source (`??` in Jupyter).

<details>
<summary>Path to the answer</summary>

1. Open [Deep Agents overview](https://docs.langchain.com/oss/python/deepagents/overview) (or run `??create_deep_agent` and follow `FilesystemMiddleware`).
2. Default: `ls`, `read_file`, `write_file`, `edit_file`, `delete`, `glob`, `grep` (filesystem) and `task` (subagents). You saw them in `deep_script.bound_tools`.
3. `execute` is added only when the backend supports execution (a sandbox, Lesson 11.09).

</details>

## ✅ Checkpoint

- [ ] I can name the four harness areas and point to the code that implements each one in my mini-harness.
- [ ] My tools change state with `Command(update=...)` and answer the call with a `ToolMessage`.
- [ ] I always pass the model to `create_deep_agent` (through `build_deep_agent`).
- [ ] I can explain why a deep agent's per-call input is bigger, and when that is worth it.
- [ ] Restart & Run All passes without keys; outputs cleared before saving.

## 🔎 Research checkpoint

Open the [Deep Agents changelog](https://docs.langchain.com/oss/python/deepagents/changelog-py) and the [releases](https://github.com/langchain-ai/deepagents/releases). Log in [`docs/research-log.md`](../../docs/research-log.md): every release since 0.7.23 (2026-10-16: the latest is 0.7.26), any breaking change or new deprecation, and whether 1.0 has shipped. Two primary sources, each with the version it applies to, and what changes in your code.

## Summary

- A harness is prompts + tools + middleware + state; you built one in ~100 lines.
- `create_deep_agent` returns a compiled LangGraph graph with filesystem, subagent, summarization and patching middleware; planning is opt-in.
- Always pass the model explicitly; the course does it in `mastery.deep.build_deep_agent`, with a rate limiter, call budgets and optional fallbacks.
- Every step re-sends the system prompt and tool schemas: steps are the free-tier budget.

---
## ➡️ What's next

**You're done with this lesson when** every box in the ✅ Checkpoint is ticked and **Kernel → Restart Kernel and Run All Cells…** finishes without errors.

1. Clear outputs before you save (**Edit → Clear Outputs of All Cells**), then **File → Save** (`Ctrl + S`).
2. Optional: log your research checkpoint in [`docs/research-log.md`](../../docs/research-log.md).
3. Next: **Lesson 11-02-models-profiles-and-budgets.ipynb** → [`11-02-models-profiles-and-budgets.ipynb`](11-02-models-profiles-and-budgets.ipynb).

[Module 11 README](README.md) · [Course outline](../../COURSE_OUTLINE.md)